# Step 13a -- CARE-Sim Selected Causal Control

Runs the updated selected-set control layer on Google Colab using:
- selected causal CARE-Sim (`caresim_selected_causal`)
- handcrafted dense reward
- terminal readmission reward model
- finite-horizon control with a hard `5`-step rollout cap

## Before running

**Step 1:** From the repo root, rebuild the upload zip locally:
```
python scripts/prepare_colab_upload.py
```

**Step 2:** Upload to Google Drive:
- `caresim_colab.zip` -> `MyDrive/CareAI/`
- `rl_dataset_selected.parquet` -> `MyDrive/CareAI/data/`
- trained selected causal CARE-Sim folder `models/icu_readmit/caresim_selected_causal/` -> `MyDrive/CareAI/models/icu_readmit/caresim_selected_causal/`
- terminal readmission model folder `models/icu_readmit/terminal_readmit_selected/` -> `MyDrive/CareAI/models/icu_readmit/terminal_readmit_selected/`

**Step 3:** Runtime -> Change runtime type -> **T4 GPU**.

This Step 13a track should be interpreted as **short-horizon finite-horizon control**, not full natural-length ICU episode optimization.


In [ ]:
# Cell 1: Mount Drive + check GPU
from google.colab import drive
drive.mount('/content/drive')

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU:   {torch.cuda.get_device_name(0)}')
    print(f'VRAM:  {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected. Step 13a will be much slower on CPU.')


In [ ]:
# Cell 2: Unzip source files + verify inputs
import os, sys, zipfile, shutil

DRIVE_ROOT = '/content/drive/MyDrive/CareAI'
ZIP_PATH   = os.path.join(DRIVE_ROOT, 'caresim_colab.zip')
UNZIP_DIR  = '/content/caresim_src'
SRC_PATH   = os.path.join(UNZIP_DIR, 'src')

DATA_PATH           = os.path.join(DRIVE_ROOT, 'data', 'rl_dataset_selected.parquet')
CARESIM_MODEL_DIR   = os.path.join(DRIVE_ROOT, 'models', 'icu_readmit', 'caresim_selected_causal')
TERMINAL_MODEL_DIR  = os.path.join(DRIVE_ROOT, 'models', 'icu_readmit', 'terminal_readmit_selected')
CONTROL_MODEL_DIR   = os.path.join(DRIVE_ROOT, 'models', 'icu_readmit', 'caresim_control_selected_causal')
REPORT_DIR          = os.path.join(DRIVE_ROOT, 'reports', 'icu_readmit', 'caresim_control_selected_causal')

CONTROL_SCRIPT = os.path.join(UNZIP_DIR, 'scripts', 'icu_readmit', 'step_13a_caresim_control.py')

for path, name in [
    (ZIP_PATH, 'caresim_colab.zip'),
    (DATA_PATH, 'rl_dataset_selected.parquet'),
    (CARESIM_MODEL_DIR, 'caresim_selected_causal'),
    (TERMINAL_MODEL_DIR, 'terminal_readmit_selected'),
]:
    if not os.path.exists(path):
        raise FileNotFoundError(f'{name} not found at {path}')

if os.path.exists(UNZIP_DIR):
    shutil.rmtree(UNZIP_DIR)

print(f'Unzipping {ZIP_PATH} ...')
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    zf.extractall(UNZIP_DIR)
print(f'Unzipped to {UNZIP_DIR}')

for path in [SRC_PATH, CONTROL_SCRIPT]:
    if not os.path.exists(path):
        raise FileNotFoundError(f'Missing expected extracted file: {path}')

os.makedirs(CONTROL_MODEL_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

print('Inputs ready:')
print(f'  Data:      {DATA_PATH}')
print(f'  CARE-Sim:  {CARESIM_MODEL_DIR}')
print(f'  Terminal:  {TERMINAL_MODEL_DIR}')
print(f'  Control:   {CONTROL_MODEL_DIR}')
print(f'  Reports:   {REPORT_DIR}')


In [ ]:
# Cell 3: Shared helpers + selected control config
import subprocess

USE_TERMINAL_READMIT_REWARD = True
ROLL_OUT_STEPS = 5
HISTORY_LEN = 5
OBS_WINDOW = 5
PLANNER_HORIZON = 3
UNCERTAINTY_PENALTY = 0.25
TERMINAL_REWARD_SCALE = 15.0

TRAIN_EPISODES = 2000
TRAIN_STEPS = 20000
BATCH_SIZE = 64
TARGET_SYNC_EVERY = 200
EPSILON_END = 0.10
EPSILON_DECAY_STEPS = 20000

ENV = {**os.environ, 'PYTHONPATH': SRC_PATH, 'PYTHONUNBUFFERED': '1'}

def run_streaming(cmd, env):
    proc = subprocess.Popen(
        cmd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    return proc.returncode

def common_args(mode):
    args = [
        sys.executable, '-u', CONTROL_SCRIPT, mode,
        '--data', DATA_PATH,
        '--model-dir', CARESIM_MODEL_DIR,
        '--history-len', str(HISTORY_LEN),
        '--observation-window', str(OBS_WINDOW),
        '--rollout-steps', str(ROLL_OUT_STEPS),
        '--planner-horizon', str(PLANNER_HORIZON),
        '--uncertainty-penalty', str(UNCERTAINTY_PENALTY),
        '--device', device,
        '--use-severity-reward',
        '--severity-mode', 'handcrafted',
    ]
    if USE_TERMINAL_READMIT_REWARD:
        args += [
            '--use-terminal-readmit-reward',
            '--terminal-model-dir', TERMINAL_MODEL_DIR,
            '--terminal-reward-scale', str(TERMINAL_REWARD_SCALE),
        ]
    return args

print('Selected Step 13a config:')
print(f'  rollout_steps:      {ROLL_OUT_STEPS}')
print(f'  planner_horizon:    {PLANNER_HORIZON}')
print(f'  reward:             handcrafted severity')
print(f'  terminal reward:    {USE_TERMINAL_READMIT_REWARD}')
print(f'  control model dir:  {CONTROL_MODEL_DIR}')
print(f'  report dir:         {REPORT_DIR}')


In [ ]:
# Cell 4: Planner-only evaluation
PLANNER_ARGS = common_args('planner') + [
    '--report-dir', REPORT_DIR,
    '--episodes-per-split', '50',
]

print('=' * 60)
print('STEP 13a -- Selected Planner Baseline')
print('=' * 60)
rc = run_streaming(PLANNER_ARGS, env=ENV)
if rc != 0:
    raise RuntimeError('Planner baseline FAILED.')
print('Planner baseline complete.')


In [ ]:
# Cell 5: DDQN training against selected CARE-Sim
DDQN_ARGS = common_args('train-ddqn') + [
    '--control-model-dir', CONTROL_MODEL_DIR,
    '--report-dir', REPORT_DIR,
    '--train-episodes', str(TRAIN_EPISODES),
    '--train-steps', str(TRAIN_STEPS),
    '--batch-size', str(BATCH_SIZE),
    '--target-sync-every', str(TARGET_SYNC_EVERY),
    '--epsilon-end', str(EPSILON_END),
    '--epsilon-decay-steps', str(EPSILON_DECAY_STEPS),
    '--warmup-steps', '256',
    '--replay-capacity', '20000',
]

print('=' * 60)
print('STEP 13a -- Selected DDQN Training')
print('=' * 60)
rc = run_streaming(DDQN_ARGS, env=ENV)
if rc != 0:
    raise RuntimeError('DDQN training FAILED.')
print('DDQN training complete.')


In [ ]:
# Cell 6: Train simple multi-armed bandit baseline
BANDIT_ARGS = common_args('train-bandit') + [
    '--control-model-dir', CONTROL_MODEL_DIR,
    '--report-dir', REPORT_DIR,
    '--train-episodes', str(TRAIN_EPISODES),
    '--train-steps', str(TRAIN_STEPS),
]

print('=' * 60)
print('STEP 13a -- Selected Bandit Training')
print('=' * 60)
rc = run_streaming(BANDIT_ARGS, env=ENV)
if rc != 0:
    raise RuntimeError('Bandit training FAILED.')
print('Bandit training complete.')


In [ ]:
# Cell 7: Evaluate planner vs DDQN vs bandit baselines
DDQN_PATH = os.path.join(CONTROL_MODEL_DIR, 'ddqn_model.pt')
BANDIT_PATH = os.path.join(CONTROL_MODEL_DIR, 'bandit_model.json')
if not os.path.exists(DDQN_PATH):
    raise FileNotFoundError(f'DDQN checkpoint not found at {DDQN_PATH}')
if not os.path.exists(BANDIT_PATH):
    raise FileNotFoundError(f'Bandit model not found at {BANDIT_PATH}')

EVAL_ARGS = common_args('eval') + [
    '--report-dir', REPORT_DIR,
    '--ddqn-path', DDQN_PATH,
    '--bandit-path', BANDIT_PATH,
    '--episodes-per-split', '100',
]

print('=' * 60)
print('STEP 13a -- Selected Policy Evaluation')
print('=' * 60)
rc = run_streaming(EVAL_ARGS, env=ENV)
if rc != 0:
    raise RuntimeError('Step 13a evaluation FAILED.')

summary_path = os.path.join(REPORT_DIR, 'step_13a_summary.json')
print('\nSaved summary:')
print(summary_path)
print(open(summary_path, 'r', encoding='utf-8').read())


In [ ]:
# Cell 8: Verify outputs on Drive
print('Control model files saved to Drive:')
for root, dirs, files in os.walk(CONTROL_MODEL_DIR):
    for f in sorted(files):
        path = os.path.join(root, f)
        size_kb = os.path.getsize(path) / 1024
        rel = os.path.relpath(path, CONTROL_MODEL_DIR)
        print(f'  {rel:55s}  {size_kb:8.1f} KB')

print('\nReport files saved to Drive:')
for root, dirs, files in os.walk(REPORT_DIR):
    for f in sorted(files):
        path = os.path.join(root, f)
        size_kb = os.path.getsize(path) / 1024
        rel = os.path.relpath(path, REPORT_DIR)
        print(f'  {rel:55s}  {size_kb:8.1f} KB')

print('\nDownload from Drive to these local repo folders:')
print('  CareAI/models/icu_readmit/caresim_control_selected_causal/')
print('  CareAI/reports/icu_readmit/caresim_control_selected_causal/')
